In [ ]:
!pip install streamlit plotly pyngrok -q
print("✓ Semua dependencies terinstall")

In [ ]:
import os

AZURE_KEY      = "C0SG9EyACffSlTnabV0ClNFyn2FGD1rb0yoHkEMHtmQuO91l7Wm8JQQJ99CDACNns7RXJ3w3AAABACOGl0AL"          # ← Paste API Key Azure kamu
AZURE_ENDPOINT = "https://projekjudol.openai.azure.com/"
AZURE_DEPLOY   = "gpt-4o"

os.environ["AZURE_KEY"]      = AZURE_KEY
os.environ["AZURE_ENDPOINT"] = AZURE_ENDPOINT
os.environ["AZURE_DEPLOY"]   = AZURE_DEPLOY

if AZURE_KEY and AZURE_ENDPOINT:
    print(f"✓ Azure credentials siap")
    print(f"  Endpoint : {AZURE_ENDPOINT}")
    print(f"  API Key  : {AZURE_KEY[:8]}...{AZURE_KEY[-4:]} (masked)")
else:
    print("⚠️  Azure credentials kosong — dashboard tetap jalan, tombol AI pakai fallback statis")


In [ ]:
import os

# Create .streamlit directory if it doesn't exist
os.makedirs('.streamlit', exist_ok=True)

# Write secrets to .streamlit/secrets.toml
secrets_path = '.streamlit/secrets.toml'
with open(secrets_path, 'w') as f:
    f.write(f'AZURE_KEY = "{os.environ["AZURE_KEY"]}"\n')
    f.write(f'AZURE_ENDPOINT = "{os.environ["AZURE_ENDPOINT"]}"\n')
    f.write(f'AZURE_DEPLOY = "{os.environ["AZURE_DEPLOY"]}"\n')

print(f"✓ {secrets_path} created with Azure credentials.")

In [ ]:
import os

print("Cek file yang dibutuhkan:")
required = {
    '06_dashboard.py'                          : 'File dashboard',
    'risk_scores_with_explanation.csv'    : 'Risk scores + GPT explanation',
    'judolguard_.csv'             : 'Features dataset',
    'model_metrics.txt'                   : 'Model metrics',
    'eda_behavioral_shift.png'            : 'EDA chart',
    'xgb_judolguard.pkl'                : 'Trained model',
}
all_ok = True
for path, desc in required.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {'✓' if ok else '✗ MISSING'} {path}  ({desc})")

if not all_ok:
    print("\n⚠️  Ada file yang missing. Jalankan notebook sebelumnya dulu.")
else:
    print("\n✓ Semua file siap. Lanjut ke Cell 3 untuk deploy.")

In [ ]:
import os
print("Working directory:", os.getcwd())
print("\nIsi folder data/:")
os.makedirs('data', exist_ok=True)
print(os.listdir('data'))

In [ ]:
# Jalankan ini di session yang sama dengan deploy
from google.colab import files

# Upload judolguard_features.csv dulu
uploaded = files.upload()
# Pilih file judolguard_features.csv dari laptop kamu

import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Pindahkan ke folder data/
import shutil
for fname in uploaded.keys():
    shutil.move(fname, f'data/{fname}')
    print(f"✓ {fname} dipindah ke data/")

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import IsolationForest
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv('/content/data/judolguard.csv')

FEATURE_COLS = [
    'hour_of_day', 'is_night', 'night_ratio_7d', 'night_ratio_14d', 'temporal_shift',
    'amount_log', 'amount_vs_avg_7d', 'total_amount_7d',
    'tx_count_24h', 'tx_count_7d', 'burst_score',
    'unique_recv_7d', 'unique_recv_24h', 'qris_ratio_7d',
    'drain_cycle_flag', 'dormant_flag'
]

X = df[FEATURE_COLS].fillna(0)
y = df['is_at_risk']

# Isolation Forest
iso = IsolationForest(n_estimators=200, contamination=0.35, random_state=42)
iso.fit(X[y==0])
raw = iso.score_samples(X)
df['anomaly_score'] = 1 - ((raw - raw.min()) / (raw.max() - raw.min()))

X['anomaly_score'] = df['anomaly_score']

# XGBoost
spw = len(y[y==0]) / len(y[y==1])
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=spw, random_state=42, n_jobs=-1,
    eval_metric='aucpr', use_label_encoder=False
)
xgb_model.fit(X, y, verbose=False)

# Risk scores per akun
df['risk_prob']  = xgb_model.predict_proba(X)[:, 1]
df['risk_score'] = (df['risk_prob'] * 100).round(1)

def get_level(s):
    if s <= 30: return 'Low'
    elif s <= 60: return 'Medium'
    elif s <= 80: return 'High'
    else: return 'Critical'

def get_rec(l):
    return {
        'Low'     : 'Monitor pasif — tidak ada tindakan segera',
        'Medium'  : 'Kirim notifikasi edukasi ke nasabah',
        'High'    : 'Batasi nominal transfer harian, minta konfirmasi',
        'Critical': 'Eskalasi ke tim compliance & flag ke OJK'
    }[l]

def get_trigger(row):
    triggers = {
        'Aktivitas malam tinggi' : row['avg_night_ratio'],
        'Frekuensi tinggi'       : row['avg_tx_24h'] / 20,
        'Banyak penerima'        : row['avg_unique_recv'] / 10,
        'Velocity burst'         : row['avg_burst_score'] / 5,
        'Pergeseran ke malam'    : max(row['avg_temporal_shift'], 0),
        'Penggunaan QRIS tinggi' : row['avg_qris_ratio'],
    }
    top = sorted(triggers.items(), key=lambda x: x[1], reverse=True)[:2]
    return ' & '.join([t[0] for t in top])

risk = df.groupby('account_id').agg(
    risk_score_max       = ('risk_score', 'max'),
    profile              = ('profile', 'first'),
    is_at_risk_true      = ('is_at_risk', 'first'),
    n_transactions       = ('step', 'count'),
    avg_night_ratio      = ('night_ratio_7d', 'mean'),
    avg_tx_24h           = ('tx_count_24h', 'mean'),
    avg_unique_recv      = ('unique_recv_7d', 'mean'),
    avg_burst_score      = ('burst_score', 'mean'),
    avg_temporal_shift   = ('temporal_shift', 'mean'),
    avg_qris_ratio       = ('qris_ratio_7d', 'mean'),
).reset_index()

risk['final_risk_score'] = risk['risk_score_max'].round(1)
risk['risk_level']       = risk['final_risk_score'].apply(get_level)
risk['recommendation']   = risk['risk_level'].apply(get_rec)
risk['top_triggers']     = risk.apply(get_trigger, axis=1)
risk['explanation']      = 'Penjelasan belum di-generate.'

risk.to_csv('data/risk_scores.csv', index=False)
risk.to_csv('data/risk_scores_with_explanation.csv', index=False)

import os; os.makedirs('models', exist_ok=True)
joblib.dump(xgb_model, 'models/xgb_judolguard.pkl')

print(f"✓ risk_scores.csv tersimpan: {len(risk)} akun")
print(f"✓ Distribusi:\n{risk['risk_level'].value_counts().to_string()}")

In [ ]:
from pyngrok import conf, ngrok
import subprocess, threading, time, os, socket

# ← Ganti dengan authtoken dari dashboard.ngrok.com/get-started/your-authtoken
conf.get_default().auth_token = "3D4RffEbfxuNhdyDlTAYgoAUQzd_wtYvdDSBVDtHNPqJvpHB"

# Kill streamlit dan ngrok yang mungkin masih jalan
os.system("pkill -f streamlit 2>/dev/null")
os.system("pkill -f ngrok 2>/dev/null") # Add this line to kill ngrok processes
time.sleep(2)

# Define paths for Streamlit logs
streamlit_stdout_log = "streamlit_stdout.log"
streamlit_stderr_log = "streamlit_stderr.log"

# Function to print Streamlit logs for debugging
def print_streamlit_logs():
    print("\nStreamlit did not become ready. Checking logs:")
    if os.path.exists(streamlit_stdout_log):
        with open(streamlit_stdout_log, "r") as f:
            print("\n--- Streamlit STDOUT ---")
            print(f.read())
    if os.path.exists(streamlit_stderr_log):
        with open(streamlit_stderr_log, "r") as f:
            print("\n--- Streamlit STDERR ---")
            print(f.read())
    print("\nConsider debugging 06_dashboard.py or increasing the timeout.")

# Jalankan streamlit di background dan redirect output ke file log
def run_streamlit_with_logs():
    with open(streamlit_stdout_log, "w") as fout, open(streamlit_stderr_log, "w") as ferr:
        subprocess.Popen([
            'streamlit', 'run', '06_dashboard.py',
            '--server.port', '8501',
            '--server.headless', 'true',
            '--server.enableCORS', 'false',
            '--server.enableXsrfProtection', 'false'
        ], env={**os.environ}, stdout=fout, stderr=ferr)

time.sleep(1) # Give a moment for pkill to settle
threading.Thread(target=run_streamlit_with_logs, daemon=True).start()

# Tunggu streamlit siap dengan polling cerdas
print("Menunggu Streamlit siap", end="")
for _ in range(60): # Increased timeout for robustness
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        # Explicitly use 127.0.0.1 for connection check
        s = socket.create_connection(("127.0.0.1", 8501), timeout=1)
        s.close()
        print(" ✓")
        break
    except:
        pass
else:
    print("\n☢☢  Timeout — tetap coba buat tunnel...")
    print_streamlit_logs()

# Buat tunnel
try:
    # ngrok.disconnect_all() # This function seems to be causing an error
    # Explicitly connect to 127.0.0.1:8501
    url = ngrok.connect("127.0.0.1:8501")
    url_str = url.public_url if hasattr(url, "public_url") else str(url)
    print(f"\n\n{'='*50}")
    print(f"  ✓ DASHBOARD LIVE!")
    print(f"  🌐 {url_str}")
    print(f"{'='*50}")
    print(f"\n  Jangan tutup cell ini selama demo.")
except Exception as e:
    print(f"\n✘ Error ngrok: {e}")
    print("  Coba restart runtime Colab, lalu jalankan ulang dari Cell 1")
    print_streamlit_logs()


In [ ]:
import os
from openai import AzureOpenAI

# Initialize the Azure OpenAI client
client = AzureOpenAI(
    api_key=os.environ.get("AZURE_KEY"),
    azure_endpoint=os.environ.get("AZURE_ENDPOINT"),
    api_version="2024-02-01"
)

# Test a simple chat completion call
try:
    response = client.chat.completions.create(
        model=os.environ.get("AZURE_DEPLOY"), # deployment name
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Hello, what is your name?"}
        ]
    )
    print("Azure OpenAI connection successful!")
    print("Response:", response.choices[0].message.content)
except Exception as e:
    print(f"Azure OpenAI connection failed: {e}")
